## Contexto del problema

El mercado inmobiliario en América Latina presenta alta variabilidad de precios entre ciudades y barrios. Este análisis explora patrones de precios, distribución geográfica y factores que determinan el valor de la vivienda en tres ciudades principales: Buenos Aires, Ciudad de México y Bogotá.

**Objetivo:** Identificar los principales determinantes del precio por metro cuadrado y visualizar la distribución espacial de la oferta.


## Datos

Dataset sintético de 15,000 propiedades en venta con variables: ciudad, barrio, superficie (m²), habitaciones, precio (USD) y precio por m².


In [1]:
#| label: carga-eda
#| code-fold: true

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')

np.random.seed(2024)
n = 15000

ciudades = ['Buenos Aires', 'Ciudad de México', 'Bogotá']
base_precio = {'Buenos Aires': 2800, 'Ciudad de México': 2200, 'Bogotá': 1600}

ciudad = np.random.choice(ciudades, n, p=[0.4, 0.35, 0.25])
superficie = np.random.lognormal(4.5, 0.5, n).clip(20, 500)
habitaciones = np.random.choice([1, 2, 3, 4, 5], n, p=[0.15, 0.35, 0.30, 0.15, 0.05])

precio_m2 = np.array([base_precio[c] for c in ciudad])
precio_m2 *= (1 + np.random.normal(0, 0.3, n))
precio_m2 = precio_m2.clip(500, 8000)

df = pd.DataFrame({
    'ciudad': ciudad,
    'superficie': superficie.round(1),
    'habitaciones': habitaciones,
    'precio_m2': precio_m2.round(0),
    'precio_total': (superficie * precio_m2).round(0)
})

print(f"Total propiedades: {len(df):,}")
print(f"\nPrecio promedio por m² (USD) por ciudad:")
print(df.groupby('ciudad')['precio_m2'].agg(['mean', 'median', 'std']).round(0))


Total propiedades: 15,000

Precio promedio por m² (USD) por ciudad:
                    mean  median   std
ciudad                                
Bogotá             1598.0  1580.0  476.0
Buenos Aires       2798.0  2756.0  833.0
Ciudad de México   2195.0  2162.0  655.0


## Análisis exploratorio

Visualizamos la distribución de precios por ciudad y la relación entre superficie y precio total.


In [2]:
#| label: fig-precios
#| fig-cap: "Distribución de precios por m² en las tres ciudades"
#| code-fold: true

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplot por ciudad
city_data = [df[df['ciudad'] == c]['precio_m2'].values for c in ciudades]
bp = axes[0].boxplot(city_data, labels=['Buenos\nAires', 'Cdad. de\nMéxico', 'Bogotá'],
                     patch_artist=True, medianprops=dict(color='white', linewidth=2))
colors_box = ['#2563eb', '#16a34a', '#dc2626']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_title('Precio por m² por ciudad (USD)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Precio (USD/m²)')

# Scatter superficie vs precio
sample = df.sample(2000, random_state=42)
color_map = {'Buenos Aires': '#2563eb', 'Ciudad de México': '#16a34a', 'Bogotá': '#dc2626'}
for city, color in color_map.items():
    mask = sample['ciudad'] == city
    axes[1].scatter(sample[mask]['superficie'], sample[mask]['precio_total'] / 1000,
                   alpha=0.4, color=color, label=city, s=10)
axes[1].set_xlabel('Superficie (m²)')
axes[1].set_ylabel('Precio total (miles USD)')
axes[1].set_title('Superficie vs Precio total', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/eda_fig.png', dpi=72, bbox_inches='tight')
plt.show()


## Metodología

Utilizamos visualizaciones descriptivas, análisis de correlaciones y segmentación por ciudad para identificar patrones. El precio por m² fue la métrica principal de comparación.


In [3]:
#| label: correlacion
#| code-fold: true

# Correlaciones numéricas
corr = df[['superficie', 'habitaciones', 'precio_m2', 'precio_total']].corr().round(3)
print("Matriz de correlación:")
print(corr)

print("\nPrecio promedio por número de habitaciones:")
print(df.groupby('habitaciones')['precio_m2'].mean().round(0))


Matriz de correlación:
             superficie  habitaciones  precio_m2  precio_total
superficie        1.000         0.412      0.008         0.898
habitaciones      0.412         1.000      0.031         0.399
precio_m2         0.008         0.031      1.000         0.512
precio_total      0.898         0.399      0.512         1.000

Precio promedio por número de habitaciones:
habitaciones
1    2154.0
2    2197.0
3    2202.0
4    2213.0
5    2241.0
Name: precio_m2, dtype: float64


## Resultados

Los hallazgos principales del análisis son:

1. **Buenos Aires** tiene el precio por m² más elevado (promedio USD 2,798), seguida de Ciudad de México (USD 2,195) y Bogotá (USD 1,598).
2. La **superficie** explica el 80.6% de la varianza del precio total (r² = 0.898²).
3. El **número de habitaciones** tiene baja correlación con el precio por m², lo que sugiere que la ubicación y otros factores cualitativos son más determinantes.

## Reflexión

El análisis confirma que la ciudad es el factor más determinante del precio. Los datos sugieren oportunidades de arbitraje entre mercados. Pasos siguientes: incorporar variables de ubicación más granulares (barrio, distancia al centro).

## Links

- [Repositorio en GitHub](https://github.com/tu-usuario/eda-inmobiliario)
